In [1]:
!pip install dotenv
!pip install openai

In [2]:
import os
import dotenv

# import torch
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
dotenv.load_dotenv(override=True)


True

In [3]:
from openai import OpenAI, AzureOpenAI
import random
import os
import time
from typing import Any
from abc import ABC, abstractmethod
import random
import time
import openai
from typing import Any


class BaseModel(ABC):
    def __init__(self, model_name: str, **kwargs):
        self._name = model_name
        self.max_tokens = kwargs.get('max_tokens', 512)
        self.temperature = kwargs.get('temperature', 0.7)
        self.top_p = kwargs.get('top_p', 1.0)
        self.reasoning_effort = kwargs.get('reasoning_effort', None)
        self.n = kwargs.get('n', 1)
        self.input_tokens = 0
        self.output_tokens = 0

    @abstractmethod
    def generate(self, messages) -> str:
        pass

    @property
    def name(self) -> str:
        return self._name
    
    def from_text_to_tokens(self, text: str) -> list[int]:
        """Convert text to tokens."""
        raise NotImplementedError("This method should be implemented by subclasses.")
    
    def from_token_to_text(self, token: int) -> str:
        """Convert a token ID back to text."""
        raise NotImplementedError("This method should be implemented by subclasses.")


class OpenRouterError(Exception):
    """Custom exception for OpenRouter API errors"""
    pass

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

OpenRouterClient = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)


class OpenRouterModel(BaseModel):
    def __init__(self, model_name: str, **kwargs):
        super().__init__(model_name=model_name, **kwargs)

    def retry_with_exponential_backoff(  # type: ignore
        func,
        initial_delay: float = 1,
        exponential_base: float = 2,
        jitter: bool = True,
        max_retries: int = 5,
        errors: tuple[Any] = (openai.RateLimitError, OpenRouterError),
    ):
        """Retry a function with exponential backoff."""

        def wrapper(*args, **kwargs):  # type: ignore
            # Initialize variables
            num_retries = 0
            delay = initial_delay

            # Loop until a successful response or max_retries is hit or an exception is raised
            while True:
                try:
                    return func(*args, **kwargs)
                # Retry on specified errors
                # except errors as e:
                except Exception as e:
                    # Increment retries
                    num_retries += 1

                    # Check if max retries has been reached
                    # if num_retries > max_retries:
                    #     raise Exception(
                    #         f"Maximum number of retries ({max_retries}) exceeded.")

                    # Increment the delay
                    delay *= exponential_base * (1 + jitter * random.random())
                    print(f"#{num_retries} Error occurred: {e}.\n Retrying in {delay} seconds.")
                    # Sleep for the delay
                    time.sleep(delay)

        return wrapper

    @retry_with_exponential_backoff
    def generate(self, messages) -> str:
        """
        Chat completion using the chat/completions endpoint.
        Supports multi-modal inputs (text + images) for vision models.
        """
        # messages = [
        #     {"role": "system", "content": SYSTEM_PROMPT},
        #     {"role": "user", "content": prompt}
        # ]
        response = OpenRouterClient.chat.completions.create(
            model=self.name,
            messages=messages,
            # max_tokens=self.max_tokens,
            temperature=self.temperature,
            top_p=self.top_p,
            n=self.n,
        )
        
        usage = getattr(response, "usage", None)
        if usage:
            self.input_tokens += usage.prompt_tokens
            self.output_tokens += usage.completion_tokens
            print(f"Total input tokens: {self.input_tokens}, Total output tokens: {self.output_tokens}")

        # Raise OpenRouterError if we get invalid response to trigger retry
        if not response or not hasattr(response, 'choices') or not response.choices:
            raise OpenRouterError("Invalid response from OpenRouter API")

        predictions = [choice.message.content.strip(
        ) for choice in response.choices if choice.message.content.strip()]

        if len(predictions) == 0:
            raise OpenRouterError("Invalid response from OpenRouter API")

        return predictions[0]

In [4]:
!pip install together

In [5]:
from together import Together
TogetherAIClient = Together()

class TogetherAIModel(BaseModel):
    def __init__(self, model_name: str, **kwargs):
        super().__init__(model_name=model_name, **kwargs)

    def retry_with_exponential_backoff(  # type: ignore
        func,
        initial_delay: float = 1,
        exponential_base: float = 2,
        jitter: bool = True,
        max_retries: int = 5,
    ):
        """Retry a function with exponential backoff."""

        def wrapper(*args, **kwargs):  # type: ignore
            # Initialize variables
            num_retries = 0
            delay = initial_delay

            # Loop until a successful response or max_retries is hit or an exception is raised
            while True:
                try:
                    return func(*args, **kwargs)
                except openai.InvalidRequestError as e:
                    raise e
                except Exception as e:
                    num_retries += 1
                    delay *= exponential_base * (1 + jitter * random.random())
                    print(f"#{num_retries} Error occurred: {e}.\n Retrying in {delay} seconds.")
                    # Sleep for the delay
                    time.sleep(delay)

        return wrapper

    @retry_with_exponential_backoff
    def generate(self, messages) -> str:
        """
        Chat completion using the chat/completions endpoint.
        Supports multi-modal inputs (text + images) for vision models.
        """
        response = TogetherAIClient.chat.completions.create(
            model=self.name,
            messages=messages,
            # max_tokens=self.max_tokens,
            max_new_tokens=1024,
            temperature=self.temperature,
            top_p=self.top_p,
            n=self.n,
        )
        
        usage = getattr(response, "usage", None)
        if usage:
            self.input_tokens += usage.prompt_tokens
            self.output_tokens += usage.completion_tokens
            print(f"Total input tokens: {self.input_tokens}, Total output tokens: {self.output_tokens}")

        # Raise OpenRouterError if we get invalid response to trigger retry
        if not response or not hasattr(response, 'choices') or not response.choices:
            raise ValueError("Zero response from Together API")

        predictions = [choice.message.content.strip(
        ) for choice in response.choices if choice.message.content.strip()]

        if len(predictions) == 0:
            raise OpenRouterError("Empty responses from Together API")

        return predictions[0]

In [6]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-70B-free"

In [7]:

# agent = CodeGenerator("google/gemma-3-1b-it")
# agent = UnslothCodeGenerator("Qwen/Qwen2.5-Coder-3B-Instruct")
# agent = UnslothCodeGenerator("google/gemma-3-1b-it")
# agent = OpenRouterModel("openai/gpt-oss-20b:free")
agent = TogetherAIModel(model_name)

In [8]:
# print(agent.generate("মৌলিক সংখ্যা বের করার জন্য একটি পাইথন ফাংশন লিখুন", ""))

In [ ]:
import signal

# Timeout handler
def _timeout_handler(signum, frame):
    raise TimeoutError("Execution timed out")

def evaluate_solution(solution_code: str, unit_tests: list[str], timeout_per_test: int = 5) -> int:
    # Clean solution code (optional: remove markdown fences)
    solution_code = solution_code.strip('` \n').replace('python\n', '').strip()
    
    # Prepare namespace
    namespace = {}
    
    # Execute solution code with timeout
    try:
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(timeout_per_test * len(unit_tests))  # total timeout for code + tests
        exec(solution_code, namespace)
        signal.alarm(0)
    except TimeoutError:
        print("⏱️ Timeout in solution code execution")
        return 0
    except Exception as e:
        print(f"❌ Error in solution code: {e}")
        return 0

    # Evaluate unit tests
    passed_count = 0
    for i, test_stmt in enumerate(unit_tests):
        try:
            signal.alarm(timeout_per_test)
            exec(test_stmt, namespace)
            signal.alarm(0)
            passed_count += 1
        except TimeoutError:
            print(f"⏱️ Test {i+1} timed out")
            signal.alarm(0)
        except AssertionError:
            print(f"❌ Test {i+1} failed: {test_stmt}")
            signal.alarm(0)
        except SystemExit as e:
            print(f"⚠️ SystemExit in test {i+1}: {e.code}")
            signal.alarm(0)
        except Exception as e:
            print(f"⚠️ Exception in test {i+1}: {e}")
            signal.alarm(0)

    return passed_count

In [10]:
!pip install pandas

In [11]:
import ast
import pandas as pd

def convert_csv_to_json(csv_file):
    df = pd.read_csv(csv_file, encoding='utf-8')
    df['test_list'] = df['test_list'].apply(lambda x: ast.literal_eval(ast.literal_eval(x)) if isinstance(x, str) else x)
    return df.to_dict(orient='records')

In [ ]:
def run_code(code: str):
    namespace = {}
    try:
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(60)  # total timeout for code + tests
        exec(code, namespace)
        signal.alarm(0)
    except TimeoutError:
        raise TimeoutError("Execution timed out")
    except AssertionError as e:
        raise AssertionError(f"Assertion failed: {e}")
    except SyntaxError as e:
        raise SyntaxError(f"Syntax error in code: {e}")
    except Exception as e:
        raise RuntimeError(f"Error while executing code: {repr(e)}")
    except SystemExit as e:
        raise RuntimeError(f"SystemExit occurred: {e.code}")
    except Exception as e:
        raise RuntimeError(f"Error while executing code: {repr(e)}")

# Prompts

In [ ]:
EXAMPLES = '''
>> Example 1:
> Instruction
```python
def smallest_multiple(n):
    """প্রথম n সংখ্যার ক্ষুদ্রতম গুণিতক খুঁজে বের করার জন্য একটি ফাংশন লিখুন।"""
```
> Solution
```python
def smallest_multiple(n):
    if (n<=2):
        return n
    i = n * 2
    factors = [number  for number in range(n, 1, -1) if number * 2 > n]
    while True:
        for a in factors:
            if i % a != 0:
                i += n
                break
            if (a == factors[-1] and i % a == 0):
                return i
                
def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, smallest_multiple(13), 360360)
    check(2, smallest_multiple(2), 2)
    check(3, smallest_multiple(1), 1)
```

>> Example 2:
> Instruction
```python
def add_dict(d1,d2):
    """সাধারণ কীগুলির জন্য মান যোগ করে দুটি অভিধানকে একত্রিত করার জন্য একটি ফাংশন লিখুন।"""
```
> Solution
```python
from collections import Counter

def add_dict(d1,d2):
    add_dict = Counter(d1) + Counter(d2)
    return add_dict  

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, add_dict({'a': 100, 'b': 200, 'c':300},{'a': 300, 'b': 200, 'd':400}), ({'b': 400, 'd': 400, 'a': 400, 'c': 300}))
    check(2, add_dict({'a': 500, 'b': 700, 'c':900},{'a': 500, 'b': 600, 'd':900}), ({'b': 1300, 'd': 900, 'a': 1000, 'c': 900}))
    check(3, add_dict({'a':900,'b':900,'d':900},{'a':900,'b':900,'d':900}), ({'b': 1800, 'd': 1800, 'a': 1800}))
```

>> Example 3:
> Instruction
```python
def count_Unset_Bits(n):
    """১ থেকে এন পর্যন্ত মোট আনসেট বিট গণনা করার জন্য একটি পাইথন ফাংশন লিখুন।"""
```
> Solution
```python
def count_Unset_Bits(n):
    cnt = 0;
    for i in range(1,n + 1):
        temp = i;
        while (temp):
            if (temp % 2 == 0):
                cnt += 1;
                temp = temp // 2;
    return cnt;  

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, count_Unset_Bits(2), 1)
    check(2, count_Unset_Bits(5), 4)
    check(3, count_Unset_Bits(14), 17)
```
'''

ADDITIONAL_EXAMPLES = '''
>> Example 4:
> Instruction
```python
def sum_of_square(n):
    """দ্বিপদী সহগগুলির বর্গক্ষেত্রের যোগফল খুঁজে বের করার জন্য একটি পাইথন ফাংশন লিখুন।"""
> Solution
```python
def factorial(start,end): 
    res = 1 
    for i in range(start,end + 1): 
        res *= i      
    return res
    
def sum_of_square(n): 
   return int(factorial(n + 1, 2 * n)/factorial(1, n))

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, sum_of_square(4), 70)
    check(2, sum_of_square(5), 252)
    check(3, sum_of_square(2), 6)
```

>> Example 5:
> Instruction
```python
def extract_date(url):
    """রেজেক্স ব্যবহার করে একটি ইউআরএল থেকে বছর, মাস এবং তারিখ বের করার জন্য একটি ফাংশন লিখুন।"""
```
> Solution
```python
import re
def extract_date(url):
    return re.findall(r'/(\\d{4})/(\\d{1,2})/(\\d{1,2})/', url)

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, extract_date("https://www.washingtonpost.com/news/football-insider/wp/2016/09/02/odell-beckhams-fame-rests-on-one-stupid-little-ball-josh-norman-tells-author/"), [('2016', '09', '02')])
    check(2, extract_date("https://www.indiatoday.in/movies/celebrities/story/wp/2020/11/03/odeof-sushant-singh-rajput-s-death-his-brother-in-law-shares-advice-for-fans-1749646/"), [('2020', '11', '03')])
    check(3, extract_date("https://economictimes.indiatimes.com/news/economy/2020/12/29/finance/pension-assets-under-pfrda-touch-rs-5-32-lakh-crore/articleshow/79736619.cms"), [('2020', '12', '29')])
```
'''

SYSTEM_PROMPT = '''
You are a Python programming assistant. 

The user will provide a function stub where the docstring is always written in Bangla.
Your task is to read the Bangla docstring, understand the requirement, and complete the function implementation in Python. 
Your response must be in English, not Bangla, and must only contain valid Python code. 
Do not add explanations, comments, or extra text. Just return the code solution.
Your main task is to carefully read the Bangla docstring and infer:
1. The expected parameter types
2. The expected return type
3. The correct implementation logic

Important guidelines:
1. The function signature is already provided in the instruction. Implement the function as specified.
2. Include a **main function** (using `def main:`) in your code that contains necessary unit tests or example calls to validate your function.
3. Do **not** call `main()` anywhere in your code. This will be executed externally.
4. Ensure that the function passes the provided unit tests.
'''

PROMPT_TEMPLATE = '''
{examples}

>> Your Task
> Instruction
```python
def {function_call}:
    """{instruction}"""
```

> Output Format (Strict)
```python
def {function_call}:
    # Add your implementation here

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {{test_id}}: Expected {{expected}}, got {{test_val}}"

def main():
    """ Unit tests using helper method 'check' for the function '{function_name}' """
    # Add your unit tests here
```

Now complete the python code for the function '{function_name}' and 'main' function. You should use the 'check' function for unit tests, which is helpful for debugging.
'''

LAST_FAILED_ATTEMPT = '''
>> Last failed attempt

> Response
{last_response}

> Error: {last_error}

Try to fix the previous code and mitigate errors.
'''


In [14]:
print(ADDITIONAL_EXAMPLES)


>> Example 4:
> Instruction
```python
def sum_of_square(n):
    """দ্বিপদী সহগগুলির বর্গক্ষেত্রের যোগফল খুঁজে বের করার জন্য একটি পাইথন ফাংশন লিখুন।"""
> Solution
```python
def factorial(start,end): 
    res = 1 
    for i in range(start,end + 1): 
        res *= i      
    return res

def sum_of_square(n): 
   return int(factorial(n + 1, 2 * n)/factorial(1, n))

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, sum_of_square(4), 70)
    check(2, sum_of_square(5), 252)
    check(3, sum_of_square(2), 6)
```

>> Example 5:
> Instruction
```python
def extract_date(url):
    """রেজেক্স ব্যবহার করে একটি ইউআরএল থেকে বছর, মাস এবং তারিখ বের করার জন্য একটি ফাংশন লিখুন।"""
```
> Solution
```python
import re
def extract_date(url):
    return re.findall(r'/(\d{4})/(\d{1,2})/(\d{1,2})/', url)

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {ex

In [ ]:
from pathlib import Path
from tqdm import tqdm
import re
from IPython.display import clear_output
import json
import csv

dev_set = convert_csv_to_json("dev_v2.csv")

responses = []

count = 0
success = 0
total_test_count = 0
passed_test_count = 0
total = 0

# Fixed tqdm bar at top
for item in tqdm((dev_set), desc="Generating", position=0):
    if item["id"] < 5:
        continue
    # open folder with task_id
    task_folder = Path(f"./dev_results/{model_name}")
    task_folder.mkdir(parents=True, exist_ok=True)
    
    # create a submission.json file if doesn't exist
    if not task_folder.joinpath("submission.json").exists():
        with open(task_folder/"submission.json", "w", newline="", encoding="utf-8") as f:
            json.dump([], f, ensure_ascii=False)

    with open(task_folder/"submission.json", "r", encoding="utf-8") as f:
        submission_data = json.load(f)

    if any(submission["id"] == item["id"] for submission in submission_data):
        print(f"Skipping {item['id']} as it already exists in submission.json")
        matching_submission = next((submission for submission in submission_data if submission["id"] == item["id"]), None)
        flag = (matching_submission["score"] == 1.0)
        if flag:
            # success += flag
            # total += 1
            continue

    function_call = item["instruction"].split("\n")[2].strip()
    function_name = ""
    match = re.match(r"(\w+)\s*\(", function_call)
    if match:
        function_name = match.group(1)
        
    prompt = PROMPT_TEMPLATE.format(
        instruction=item["instruction"].split("\n")[0].strip(),
        function_call=function_call,
        function_name=function_name,
        examples=EXAMPLES
        # examples=""
    )
    history = []
    attempt = 0
    default_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    record = None
    while True:       
        # if last_error is not None:
        #     prompt += "\n" + LAST_FAILED_ATTEMPT.format(
        #         last_response=response,
        #         last_error=last_error
        #     )
        print(f"======================== {item['id']}.{attempt} =========================")
        messages = default_messages + history if len(history) < 4 else history[-6:]
        
        try:
            response = agent.generate(messages)
        except Exception as e:
            break
        
        
        # Extract Python code block
        pattern = re.compile(r"```python\s+([\s\S]*?)```", re.MULTILINE)
        match = pattern.search(response)

        print(response)
        if match:
            code_inside = match.group(1)
            response = "```python\n" + code_inside + "\n```"
            history.append({"role": "assistant", "content": response})
            if re.search(rf"def\s+{function_name}\s*\(", code_inside):
                print(f"{function_name} function exists")
            else:
                print(f"No {function_name} function exists")
                last_error = "Invalid Response: No '"+function_name+"' function found. This is a required function."
                history.append({"role": "user", "content": f"{last_error}"})
                continue

            pattern = r'if __name__ == ["\']__main__["\']:\n(?:[ \t]+.*\n?)*'
            clean_code = re.sub(pattern, '', code_inside, flags=re.MULTILINE)
            record = {
                "id": item["id"],
                "response": clean_code
            }
            
            if re.search(r"def\s+main\s*\(\s*\)", code_inside):
                print("Main function exists. Processing code.")
                code_inside = clean_code + "\n\n# Call main function for testing\nmain()"

            
            try:
                run_code(code_inside)
                break
            except AssertionError as e:
                attempt += 1
                last_error = e
                print(f"Error: {e}")
                if attempt > 5:
                    break
            except SyntaxError as e:
                last_error = e
                print(f"Error: {e}")
            except Exception as e:
                last_error = e
                print(f"Error: {e}")
        else:
            print("No Python code block found.")
            last_error = "Invalid Response: No Python code block found. Please ensure your code is enclosed in a code block like:\n```python\n# your code here\n```."
            history.append({"role": "assistant", "content": response})

        history.append({"role": "user", "content": f"{last_error}"})

        with open(task_folder/f"{item['id']}.json", "w", encoding="utf-8") as f:
            json.dump(default_messages + history, f, ensure_ascii=False, indent=4)
    
    with open(task_folder/f"{item['id']}.json", "w", encoding="utf-8") as f:
        json.dump(default_messages + history, f, ensure_ascii=False, indent=4)  

    total += 1
    total_test_count += len(item["test_list"])
    if record is not None:
        responses.append(record)
        count = evaluate_solution(record["response"], item["test_list"])
        success += (count == len(item["test_list"]))
        passed_test_count += count
        # add result to a submission.json
        index = next((i for i, submission in enumerate(submission_data) if submission["id"] == item["id"]), None)
        if index is None:
            submission_data.append({"id": item["id"], "response": record["response"], "score": count/len(item["test_list"])})
        else:
            submission_data[index] = {"id": item["id"], "response": record["response"], "score": count/len(item["test_list"])}
            
        with open(task_folder/"submission.json", "w", encoding="utf-8") as f:
            json.dump(submission_data, f, ensure_ascii=False, indent=4)

    # Clear previous logs and print updated stats
    clear_output(wait=True)
    print(f"Task: {item['id']} -> Passed {count}/{len(item['test_list'])}")
    print(f"Complete: {success/total*100:.2f}%")
    print(f"Partial: {passed_test_count/total_test_count*100:.2f}%")

Generating:  58%|█████▊    | 234/400 [1:44:52<1:14:31, 26.94s/it]

Task: 234 -> Passed 1/3
Complete: 13.54%
Partial: 22.22%
Skipping 235 as it already exists in submission.json
Skipping 236 as it already exists in submission.json
======================== 236.0 =========================
Total input tokens: 403149, Total output tokens: 302714
<think>
Okay, I need to solve this problem where I have to write a Python function called max_sub_array_sum that takes a list lst and an integer n. The goal is to find the maximum sum of a sub-array of length n. Hmm, let's think about how to approach this.

First, I should understand the problem correctly. So, given a list of numbers, I need to look at all possible contiguous subarrays of length exactly n and find the one with the highest sum. Then, return that maximum sum.

Wait, but what if the list is empty or n is larger than the list length? Oh, right, I should handle those edge cases. But the problem statement probably assumes that the list is valid and n is a positive integer within the list's bounds.

So, t

In [ ]:
print(f"Accuracy: {success/total*100:.2f}%")

import json
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"submission_{timestamp}.json"

with open(filename, 'w', encoding='utf-8') as f:
    json.dump(responses, f, ensure_ascii=False, indent=2)

print(f"Submission file: {filename}")

In [ ]:
!pip install tqdm

In [ ]:
from pathlib import Path
import json
from tqdm import tqdm

dev_set = convert_csv_to_json("dev_v2_fixed.csv")
task_folder = Path(f"./dev_results/{model_name}")
with open(task_folder/"submission.json", "r", encoding="utf-8") as f:
    submission_data = json.load(f)

count = 0
success = 0
total_test_count = 0
passed_test_count = 0
total = 0

for item in tqdm(dev_set, desc="Evaluating", position=0):
    # Find the json with submission["id"] == item["id"]
    matching_submission = next((submission for submission in submission_data if submission["id"] == item["id"]), None)
    if matching_submission:
        print(f"Evaluating {item['id']}...")
        count = evaluate_solution(matching_submission["response"], item["test_list"])
        if count == len(item["test_list"]):
            print(f"✅ All tests passed for {item['id']}")
        success += (count == len(item["test_list"]))
        total_test_count += len(item["test_list"])
        passed_test_count += count
        total += 1

print(f"Accuracy (Pass@1): {success/total*100:.2f}%")
print(f"Unit Test Success: {passed_test_count/total_test_count*100:.2f}%")

Evaluating: 100%|██████████| 400/400 [00:00<00:00, 168954.84it/s]

Evaluating 1...
4
❌ Test 1 failed: assert max_chain_length([(5, 24), (15, 25),(27, 40), (50, 60)], 4) == 3
❌ Test 2 failed: assert max_chain_length([(1, 2), (3, 4),(5, 6), (7, 8)], 4) == 4
❌ Test 3 failed: assert max_chain_length([(19, 10), (11, 12),(13, 14), (15, 16), (31, 54)], 5) == 5
Evaluating 2...
Evaluating 3...
❌ Test 1 failed: assert get_ludic(10) == [1, 2, 3, 5, 7]
❌ Test 2 failed: assert get_ludic(25) == [1, 2, 3, 5, 7, 11, 13, 17, 23, 25]
❌ Test 3 failed: assert get_ludic(45) == [1, 2, 3, 5, 7, 11, 13, 17, 23, 25, 29, 37, 41, 43]
Evaluating 4...
Evaluating 5...
Evaluating 6...
Accuracy (Pass@1): 66.67%
Unit Test Success: 66.67%
